<a href="https://colab.research.google.com/github/mehnazpa/AIML_Lab/blob/main/Logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import pandas as pd

# Upload file from your device
uploaded = files.upload()

# Read the uploaded CSV file
filename = list(uploaded.keys())[0]
data = pd.read_csv("employee.csv")

print("File uploaded successfully!")
print(data.head())

In [ ]:

data["High_Salary"] = (data["Salary"] >= 100000).astype(int)

print("\nTarget variable created:")
print(data[["Salary", "High_Salary"]].head())


X = data[
    [
        "Age",
        "Gender",
        "Department",
        "Job_Title",
        "Experience_Years",
        "Education_Level",
        "Location"
    ]
]

y = data["High_Salary"]


X = pd.get_dummies(X, drop_first=True)


from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)


y_pred = model.predict(X_test)


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("\n========== MODEL PERFORMANCE ==========")

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))





Target variable created:
   Salary  High_Salary
0   90000            0
1  195000            1
2   35000            0
3   75000            0
4   70000            0

========== MODEL PERFORMANCE ==========
Accuracy : 0.986
Precision: 0.9916247906197655
Recall   : 0.9850249584026622
F1 Score : 0.988313856427379

Confusion Matrix:
[[ 788   10]
 [  18 1184]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       798
           1       0.99      0.99      0.99      1202

    accuracy                           0.99      2000
   macro avg       0.98      0.99      0.99      2000
weighted avg       0.99      0.99      0.99      2000



In [ ]:
print("\n========== REAL-TIME PREDICTION ==========")

age = float(input("Enter Age: "))
gender = input("Enter Gender: ")
department = input("Enter Department: ")
job_title = input("Enter Job Title: ")
experience = float(input("Enter Experience Years: "))
education = input("Enter Education Level: ")
location = input("Enter Location: ")


new_employee = pd.DataFrame({
    "Age": [age],
    "Gender": [gender],
    "Department": [department],
    "Job_Title": [job_title],
    "Experience_Years": [experience],
    "Education_Level": [education],
    "Location": [location]
})

new_employee = pd.get_dummies(new_employee)

new_employee = new_employee.reindex(
    columns=X.columns,
    fill_value=0
)

prediction = model.predict(new_employee)
probability = model.predict_proba(new_employee)

print("\nProbability of Low Salary :", probability[0][0])
print("Probability of High Salary:", probability[0][1])

if prediction[0] == 1:
    print("Prediction: HIGH SALARY")
else:
    print("Prediction: LOW SALARY")


========== REAL-TIME PREDICTION ==========
Enter Age: 54
Enter Gender: Male
Enter Department: HR
Enter Job Title: Analyst
Enter Experience Years: 15
Enter Education Level: PHD
Enter Location: New York

Probability of Low Salary : 1.4651884050476127e-07
Probability of High Salary: 0.9999998534811595
Prediction: HIGH SALARY


In [ ]:


import pandas as pd
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


data = pd.read_csv("employee.csv")


data["High_Salary"] = (data["Salary"] >= 100000).astype(int)


X = data[
    [
        "Age",
        "Gender",
        "Department",
        "Job_Title",
        "Experience_Years",
        "Education_Level",
        "Location"
    ]
]

y = data["High_Salary"]


X = pd.get_dummies(X, drop_first=True)


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

def predict_salary(
    age,
    gender,
    department,
    job_title,
    experience,
    education,
    location
):


    new_employee = pd.DataFrame({
        "Age": [age],
        "Gender": [gender],
        "Department": [department],
        "Job_Title": [job_title],
        "Experience_Years": [experience],
        "Education_Level": [education],
        "Location": [location]
    })

    new_employee = pd.get_dummies(new_employee)

    new_employee = new_employee.reindex(
        columns=X.columns,
        fill_value=0
    )

    prediction = model.predict(new_employee)


    probability = model.predict_proba(new_employee)

    low_probability = probability[0][0]
    high_probability = probability[0][1]

    if prediction[0] == 1:
        result = "HIGH SALARY"
    else:
        result = "LOW SALARY"


    return (
        result,
        f"{low_probability:.2%}",
        f"{high_probability:.2%}"
    )


with gr.Blocks() as demo:

    gr.Markdown(
        "# 💼 Employee Salary Prediction"
    )

    gr.Markdown(
        "Logistic Regression based High/Low Salary Prediction"
    )

    gr.Markdown("## 📊 Model Performance")

    gr.Markdown(
        f"""
        **Accuracy:** {accuracy:.2%}

        **Precision:** {precision:.2%}

        **Recall:** {recall:.2%}

        **F1 Score:** {f1:.2%}
        """
    )


    gr.Markdown("---")

    gr.Markdown("## 👤 Enter Employee Details")


    age = gr.Number(
        label="Age",
        value=30
    )


    gender = gr.Dropdown(
        choices=sorted(data["Gender"].unique().tolist()),
        label="Gender"
    )


    department = gr.Dropdown(
        choices=sorted(data["Department"].unique().tolist()),
        label="Department"
    )


    job_title = gr.Dropdown(
        choices=sorted(data["Job_Title"].unique().tolist()),
        label="Job Title"
    )


    experience = gr.Number(
        label="Experience Years",
        value=5
    )


    education = gr.Dropdown(
        choices=sorted(
            data["Education_Level"].unique().tolist()
        ),
        label="Education Level"
    )


    location = gr.Dropdown(
        choices=sorted(
            data["Location"].unique().tolist()
        ),
        label="Location"
    )


    predict_button = gr.Button(
        "🔮 Predict Salary"
    )

    result = gr.Textbox(
        label="Prediction"
    )

    low_probability = gr.Textbox(
        label="Low Salary Probability"
    )

    high_probability = gr.Textbox(
        label="High Salary Probability"
    )

    predict_button.click(
        fn=predict_salary,
        inputs=[
            age,
            gender,
            department,
            job_title,
            experience,
            education,
            location
        ],
        outputs=[
            result,
            low_probability,
            high_probability
        ]
    )


demo.launch(share=True)